<a href="https://colab.research.google.com/github/rudalshan0412-code/attention-is-all-you-need-pytorch/blob/main/08)_Decoder_Layer.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Google Drive 연결

from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
# 프로젝트 경로 설정

from pathlib import Path
import sys

PROJECT_ROOT = Path(
    "/content/drive/MyDrive/attention_is_all_you_need"
)

SRC_DIR = PROJECT_ROOT / "src"

PROJECT_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)

SRC_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(
        0,
        str(PROJECT_ROOT),
    )


print("PROJECT_ROOT:", PROJECT_ROOT)
print("SRC_DIR:", SRC_DIR)

PROJECT_ROOT: /content/drive/MyDrive/attention_is_all_you_need
SRC_DIR: /content/drive/MyDrive/attention_is_all_you_need/src


In [ ]:
# 현재 src 구조 확인

for path in sorted(SRC_DIR.iterdir()):
    print(path.name)

__pycache__
attention.py
decoder_layer.py
encoder.py
encoder_layer.py
feed_forward.py
mask.py
multi_head_attention.py
positional_encoding.py


In [ ]:
# 기존 MultiHeadAttention API 확인

print(
    (SRC_DIR / "multi_head_attention.py").read_text()
)


import torch
import torch.nn as nn

from src.attention import ScaledDotProductAttention


class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, num_heads):
        super().__init__()

        assert d_model % num_heads == 0

        self.d_model = d_model
        self.num_heads = num_heads
        self.head_dim = d_model // num_heads

        self.W_Q = nn.Linear(d_model, d_model)
        self.W_K = nn.Linear(d_model, d_model)
        self.W_V = nn.Linear(d_model, d_model)

        self.attention = ScaledDotProductAttention()

        self.W_O = nn.Linear(d_model, d_model)

    def split_heads(self, x):
        """
        x:
            (batch_size, seq_len, d_model)

        Returns:
            (batch_size, num_heads, seq_len, head_dim)
        """

        batch_size, seq_len, _ = x.shape

        x = x.view(
            batch_size,
            seq_len,
            self.num_heads,
            self.head_dim,
        )

        x = x.transpose(1, 2)

        return x


In [ ]:
# 기존 Encoder Layer 확인

print(
    (SRC_DIR / "encoder_layer.py").read_text()
)


import torch.nn as nn

from src.multi_head_attention import MultiHeadAttention
from src.feed_forward import PositionwiseFeedForward


class EncoderLayer(nn.Module):
    def __init__(
        self,
        d_model,
        num_heads,
        d_ff,
        dropout=0.1,
    ):
        super().__init__()

        self.self_attention = MultiHeadAttention(
            d_model,
            num_heads,
        )

        self.feed_forward = PositionwiseFeedForward(
            d_model,
            d_ff,
        )

        self.dropout1 = nn.Dropout(dropout)
        self.dropout2 = nn.Dropout(dropout)

        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)

    def forward(self, x, mask=None):
        # 1. Multi-Head Self-Attention
        attention_output, attention_weights = self.self_attention(
            x,
            x,
            x,
            mask,
        )

        # 2. 첫 번째 Add & Norm
        attention_output = self.dropout1(attention_output)

      

In [ ]:
# 기존 Mask 확인

print(
    (SRC_DIR / "mask.py").read_text()
)


import torch


def create_padding_mask(
    token_ids,
    pad_idx,
):
    mask = token_ids != pad_idx

    mask = mask.unsqueeze(1).unsqueeze(2)

    return mask


def create_causal_mask(
    seq_len,
    device=None,
):
    mask = torch.ones(
        seq_len,
        seq_len,
        dtype=torch.bool,
        device=device,
    )

    mask = torch.tril(mask)

    mask = mask.unsqueeze(0).unsqueeze(0)

    return mask



In [ ]:
'''
Decoder Layer은 크게 3가지로 구분된다(Masked self-attention, Cross-attention, Feed Forward)

Masked self attention:
 Casual mask가 적용된 상태에서 생성할 문장 내부의 토큰들 사이의 관계를 파악

Cross-attention:
 Encdor가 입력 문장을 다듬어 내놓은 정보를 Decoder에 가져와 attention

Position wide Feed Forward Network:
 Attention 연산을 통해 정보가 섞인 각 토큰 백터에 관해 비선형 변환을 거쳐 표현력을 고도화
'''

'\nDecoder Layer은 크게 3가지로 구분된다(Masked self-attention, Cross-attention, Feed Forward)\n\nMasked self attention:\n Casual mask가 적용된 상태에서 생성할 문장 내부의 토큰들 사이의 관계를 파악\n\nCross-attention:\n Encdor가 입력 문장을 다듬어 내놓은 정보를 Decoder에 가져와 attention\n\nPosition wide Feed Forward Network:\n Attention 연산을 통해 정보가 섞인 각 토큰 백터에 관해 비선형 변환을 거쳐 표현력을 고도화\n'

In [ ]:
# decoder_layer.py 전체 코드

import torch.nn as nn

from src.multi_head_attention import MultiHeadAttention
from src.feed_forward import PositionwiseFeedForward


class DecoderLayer(nn.Module):
    def __init__(
        self,
        d_model,
        num_heads,
        d_ff,
        dropout=0.1,
    ):
        super().__init__()

        self.self_attention = MultiHeadAttention(
            d_model,
            num_heads,
        )
        # Decoder target sequence의 내부 attention(Q = K = V = x)
        # 모두 동일한 입력데이터 x로부터 생성

        self.cross_attention = MultiHeadAttention(
            d_model,
            num_heads,
        )
        # Encoder와 Decoder을 연결(Q = Decoder, K = Encoder, V = Encoder)
        # 앞선 decoder layer에서 만들어준 데이터를 Q로 사용, K와 V는 Encoder의 최종 tensor 사용
        # Self-Attention과는 별개

        self.feed_forward = PositionwiseFeedForward(
            d_model,
            d_ff,
        )
        # 각 target postion마다 독립적으로 d_model -> d_ff(더 넓은 차원으로 확장) -> d_model(원래 규격)로 변환

        self.dropout1 = nn.Dropout(dropout)
        self.dropout2 = nn.Dropout(dropout)
        self.dropout3 = nn.Dropout(dropout)

        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.norm3 = nn.LayerNorm(d_model)

    def forward(
        self,
        x,
        encoder_output,
        self_attention_mask=None,
        cross_attention_mask=None,
    ):
    # self-attention
        self_attention_output, self_attention_weights = ( # 전부 같은 데이터 x(mask 적용한) 사용
            self.self_attention(
                x,
                x,
                x,
                self_attention_mask,
            )
        )

        self_attention_output = self.dropout1( # 과적합 방지용  dropout 적용
            self_attention_output
        )

        x = self.norm1( # 정규화
            x + self_attention_output
        )
   # cross attention

        cross_attention_output, cross_attention_weights = ( # 기존 decoder layer의 output, encoder의 최종 output 사용
            self.cross_attention(
                x,
                encoder_output,
                encoder_output,
                cross_attention_mask,
            )
        )

        cross_attention_output = self.dropout2( # 과적합 방지용 dropout 적용
            cross_attention_output
        )

        x = self.norm2( # 정규화
            x + cross_attention_output
        )
    # feed-forward

        ffn_output = self.feed_forward(x)

        ffn_output = self.dropout3(
            ffn_output
        )

        x = self.norm3(
            x + ffn_output
        )

        return (
            x,
            self_attention_weights,
            cross_attention_weights,
        )

In [ ]:
# 파일 저장

%%writefile /content/drive/MyDrive/attention_is_all_you_need/src/decoder_layer.py

import torch.nn as nn

from src.multi_head_attention import MultiHeadAttention
from src.feed_forward import PositionwiseFeedForward


class DecoderLayer(nn.Module):
    def __init__(
        self,
        d_model,
        num_heads,
        d_ff,
        dropout=0.1,
    ):
        super().__init__()

        self.self_attention = MultiHeadAttention(
            d_model,
            num_heads,
        )

        self.cross_attention = MultiHeadAttention(
            d_model,
            num_heads,
        )

        self.feed_forward = PositionwiseFeedForward(
            d_model,
            d_ff,
        )

        self.dropout1 = nn.Dropout(dropout)
        self.dropout2 = nn.Dropout(dropout)
        self.dropout3 = nn.Dropout(dropout)

        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.norm3 = nn.LayerNorm(d_model)

    def forward(
        self,
        x,
        encoder_output,
        self_attention_mask=None,
        cross_attention_mask=None,
    ):
        self_attention_output, self_attention_weights = (
            self.self_attention(
                x,
                x,
                x,
                self_attention_mask,
            )
        )

        self_attention_output = self.dropout1(
            self_attention_output
        )

        x = self.norm1(
            x + self_attention_output
        )

        cross_attention_output, cross_attention_weights = (
            self.cross_attention(
                x,
                encoder_output,
                encoder_output,
                cross_attention_mask,
            )
        )

        cross_attention_output = self.dropout2(
            cross_attention_output
        )

        x = self.norm2(
            x + cross_attention_output
        )

        ffn_output = self.feed_forward(x)

        ffn_output = self.dropout3(
            ffn_output
        )

        x = self.norm3(
            x + ffn_output
        )

        return (
            x,
            self_attention_weights,
            cross_attention_weights,
        )

Overwriting /content/drive/MyDrive/attention_is_all_you_need/src/decoder_layer.py


In [ ]:
# 파일 구조 확인

for path in sorted(SRC_DIR.iterdir()):
    print(path.name)

__pycache__
attention.py
decoder_layer.py
encoder.py
encoder_layer.py
feed_forward.py
mask.py
multi_head_attention.py
positional_encoding.py


In [ ]:
# Decoder layer import

import torch

from src.decoder_layer import DecoderLayer
from src.mask import (
    create_padding_mask,
    create_causal_mask,
)

In [ ]:
# 기본 설정

batch_size = 2

source_len = 5
target_len = 4

d_model = 8
num_heads = 2
d_ff = 32

dropout = 0.0

In [ ]:
# DecoderLayer 생성

decoder_layer = DecoderLayer(
    d_model=d_model,
    num_heads=num_heads,
    d_ff=d_ff,
    dropout=dropout,
)

print(decoder_layer)

DecoderLayer(
  (self_attention): MultiHeadAttention(
    (W_Q): Linear(in_features=8, out_features=8, bias=True)
    (W_K): Linear(in_features=8, out_features=8, bias=True)
    (W_V): Linear(in_features=8, out_features=8, bias=True)
    (attention): ScaledDotProductAttention()
    (W_O): Linear(in_features=8, out_features=8, bias=True)
  )
  (cross_attention): MultiHeadAttention(
    (W_Q): Linear(in_features=8, out_features=8, bias=True)
    (W_K): Linear(in_features=8, out_features=8, bias=True)
    (W_V): Linear(in_features=8, out_features=8, bias=True)
    (attention): ScaledDotProductAttention()
    (W_O): Linear(in_features=8, out_features=8, bias=True)
  )
  (feed_forward): PositionwiseFeedForward(
    (linear1): Linear(in_features=8, out_features=32, bias=True)
    (linear2): Linear(in_features=32, out_features=8, bias=True)
  )
  (dropout1): Dropout(p=0.0, inplace=False)
  (dropout2): Dropout(p=0.0, inplace=False)
  (dropout3): Dropout(p=0.0, inplace=False)
  (norm1): LayerNo

In [ ]:
# 기본 랜덤 Tensor

torch.manual_seed(0)

x = torch.randn(
    batch_size,
    target_len,
    d_model,
)

encoder_output = torch.randn(
    batch_size,
    source_len,
    d_model,
)

# shape 확인

print("Decoder input:", x.shape)
print("Encoder output:", encoder_output.shape)

Decoder input: torch.Size([2, 4, 8])
Encoder output: torch.Size([2, 5, 8])


In [ ]:
# Mask 없이 기본 forward

output, self_attention_weights, cross_attention_weights = (
    decoder_layer(
        x,
        encoder_output,
    )

)

# shape

print(
    "Decoder output:",
    output.shape,
)

print(
    "Self-Attention weights:",
    self_attention_weights.shape,
)

print(
    "Cross-Attention weights:",
    cross_attention_weights.shape,
)

Decoder output: torch.Size([2, 4, 8])
Self-Attention weights: torch.Size([2, 2, 4, 4])
Cross-Attention weights: torch.Size([2, 2, 4, 5])


In [ ]:
# Self Attention과 Cross Attention 독립성 확인

print(
    decoder_layer.self_attention
    is decoder_layer.cross_attention
)
# False 나와야 정상

False


In [ ]:
# Parameter 공유 여부 확인

self_param_ids = {
    id(parameter)
    for parameter
    in decoder_layer.self_attention.parameters()
}

cross_param_ids = {
    id(parameter)
    for parameter
    in decoder_layer.cross_attention.parameters()
}

shared_param_ids = (
    self_param_ids
    & cross_param_ids
)

print(
    "공유 Parameter 개수:",
    len(shared_param_ids),
)

공유 Parameter 개수: 0


In [ ]:
# Target Token ID 생성

target_token_ids = torch.tensor([
    [5, 8, 3, 9],
    [7, 2, 0, 0],
])

print(target_token_ids)

tensor([[5, 8, 3, 9],
        [7, 2, 0, 0]])


In [ ]:
# Target Padding Mask

target_padding_mask = create_padding_mask(
    target_token_ids,
    pad_idx=0,
)

print(
    target_padding_mask
)

print(
    target_padding_mask.shape
)

tensor([[[[ True,  True,  True,  True]]],


        [[[ True,  True, False, False]]]])
torch.Size([2, 1, 1, 4])


In [ ]:
# Casual Mask

causal_mask = create_causal_mask(
    target_len,
)

print(causal_mask)

print(
    causal_mask.shape
)

tensor([[[[ True, False, False, False],
          [ True,  True, False, False],
          [ True,  True,  True, False],
          [ True,  True,  True,  True]]]])
torch.Size([1, 1, 4, 4])


In [ ]:
# Decoder Self-Attention Mask

self_attention_mask = (
    target_padding_mask
    & causal_mask
)

print(
    self_attention_mask
)

print(
    self_attention_mask.shape
)

tensor([[[[ True, False, False, False],
          [ True,  True, False, False],
          [ True,  True,  True, False],
          [ True,  True,  True,  True]]],


        [[[ True, False, False, False],
          [ True,  True, False, False],
          [ True,  True, False, False],
          [ True,  True, False, False]]]])
torch.Size([2, 1, 4, 4])


In [ ]:
# Source Token ID

source_token_ids = torch.tensor([
    [10, 11, 12, 13, 14],
    [20, 21, 22, 0, 0],
])

In [ ]:
# Source Padding Mask

source_padding_mask = create_padding_mask(
    source_token_ids,
    pad_idx=0,
)

print(source_padding_mask)

print(
    source_padding_mask.shape
)

tensor([[[[ True,  True,  True,  True,  True]]],


        [[[ True,  True,  True, False, False]]]])
torch.Size([2, 1, 1, 5])


In [ ]:
# 두 Mask를 DecoderLayer 전달

(
    masked_output,
    masked_self_attention_weights,
    masked_cross_attention_weights,
) = decoder_layer(
    x,
    encoder_output,
    self_attention_mask=self_attention_mask,
    cross_attention_mask=source_padding_mask,
)

# shape

print(
    "Output:",
    masked_output.shape,
)

print(
    "Self-Attention:",
    masked_self_attention_weights.shape,
)

print(
    "Cross-Attention:",
    masked_cross_attention_weights.shape,
)

Output: torch.Size([2, 4, 8])
Self-Attention: torch.Size([2, 2, 4, 4])
Cross-Attention: torch.Size([2, 2, 4, 5])


In [ ]:
# Self Attention weight 직접 확인

print(
    masked_self_attention_weights[
        0,
        0,
    ]
)

tensor([[1.0000, 0.0000, 0.0000, 0.0000],
        [0.5241, 0.4759, 0.0000, 0.0000],
        [0.1882, 0.4731, 0.3387, 0.0000],
        [0.2783, 0.2301, 0.2935, 0.1981]], grad_fn=<SelectBackward0>)


In [ ]:
# 미래 token Attention이 0인지 자동 검사

expanded_self_mask = (
    self_attention_mask.expand(
        batch_size,
        num_heads,
        target_len,
        target_len,
    )
)

blocked_self_weights = (
    masked_self_attention_weights[
        ~expanded_self_mask
    ]
)

print(
    "차단 위치 최대 Attention weight:",
    blocked_self_weights.abs().max().item(),
)

assert torch.all(
    blocked_self_weights == 0
)

print(
    "Self-Attention Mask 적용 정상"
)

차단 위치 최대 Attention weight: 0.0
Self-Attention Mask 적용 정상


In [ ]:
# Target PAD Key가 0인지 직접 확인

print(
    masked_self_attention_weights[
        1,
        :,
        :,
        2:,
    ]
)

# 자동 검사

assert torch.all(
    masked_self_attention_weights[
        1,
        :,
        :,
        2:
    ]
    == 0
)

print(
    "Target PAD Key 차단 정상"
)

tensor([[[0., 0.],
         [0., 0.],
         [0., 0.],
         [0., 0.]],

        [[0., 0.],
         [0., 0.],
         [0., 0.],
         [0., 0.]]], grad_fn=<SliceBackward0>)
Target PAD Key 차단 정상


In [ ]:
# Cross Attention Source PAD 확인

print(
    masked_cross_attention_weights[
        1,
        :,
        :,
        3:,
    ]
)

# 검사

assert torch.all(
    masked_cross_attention_weights[
        1,
        :,
        :,
        3:
    ]
    == 0
)

print(
    "Source PAD Key 차단 정상"
)

tensor([[[0., 0.],
         [0., 0.],
         [0., 0.],
         [0., 0.]],

        [[0., 0.],
         [0., 0.],
         [0., 0.],
         [0., 0.]]], grad_fn=<SliceBackward0>)
Source PAD Key 차단 정상


In [ ]:
# 반복 실행 안정성

# 현재 dropout = 0.0임으로 동일 입력을 2번 넣어도 같은 결과가 나와야한다.

output1, self_w1, cross_w1 = decoder_layer(
    x,
    encoder_output,
    self_attention_mask,
    source_padding_mask,
)

output2, self_w2, cross_w2 = decoder_layer(
    x,
    encoder_output,
    self_attention_mask,
    source_padding_mask,
)

# 검사

print(
    torch.allclose(
        output1,
        output2,
    )
)

print(
    torch.allclose(
        self_w1,
        self_w2,
    )
)

print(
    torch.allclose(
        cross_w1,
        cross_w2,
    )
)

True
True
True


In [ ]:
# 다양한 batch / sequence length 테스트

test_shapes = [
    (1, 3, 5),
    (2, 4, 6),
    (3, 5, 4),
]

for batch_size, target_len, source_len in test_shapes:

    x_test = torch.randn(
        batch_size,
        target_len,
        d_model,
    )

    encoder_test = torch.randn(
        batch_size,
        source_len,
        d_model,
    )

    (
        output_test,
        self_weights_test,
        cross_weights_test,
    ) = decoder_layer(
        x_test,
        encoder_test,
    )

    print(
        f"B={batch_size}, "
        f"T={target_len}, "
        f"S={source_len}"
    )

    print(
        " output:",
        output_test.shape,
    )

    print(
        " self:",
        self_weights_test.shape,
    )

    print(
        " cross:",
        cross_weights_test.shape,
    )

    print()

B=1, T=3, S=5
 output: torch.Size([1, 3, 8])
 self: torch.Size([1, 2, 3, 3])
 cross: torch.Size([1, 2, 3, 5])

B=2, T=4, S=6
 output: torch.Size([2, 4, 8])
 self: torch.Size([2, 2, 4, 4])
 cross: torch.Size([2, 2, 4, 6])

B=3, T=5, S=4
 output: torch.Size([3, 5, 8])
 self: torch.Size([3, 2, 5, 5])
 cross: torch.Size([3, 2, 5, 4])



In [ ]:
# 통합 테스트

import torch

from src.decoder_layer import DecoderLayer
from src.mask import (
    create_padding_mask,
    create_causal_mask,
)


torch.manual_seed(0)

batch_size = 2
target_len = 4
source_len = 5

d_model = 8
num_heads = 2
d_ff = 32


decoder_layer = DecoderLayer(
    d_model=d_model,
    num_heads=num_heads,
    d_ff=d_ff,
    dropout=0.0,
)


# --------------------------------------------------
# Attention 객체 / Parameter 독립성
# --------------------------------------------------

assert (
    decoder_layer.self_attention
    is not decoder_layer.cross_attention
)

self_param_ids = {
    id(parameter)
    for parameter
    in decoder_layer.self_attention.parameters()
}

cross_param_ids = {
    id(parameter)
    for parameter
    in decoder_layer.cross_attention.parameters()
}

assert len(
    self_param_ids
    & cross_param_ids
) == 0


# --------------------------------------------------
# 입력
# --------------------------------------------------

x = torch.randn(
    batch_size,
    target_len,
    d_model,
)

encoder_output = torch.randn(
    batch_size,
    source_len,
    d_model,
)


target_token_ids = torch.tensor([
    [5, 8, 3, 9],
    [7, 2, 0, 0],
])

source_token_ids = torch.tensor([
    [10, 11, 12, 13, 14],
    [20, 21, 22, 0, 0],
])


# --------------------------------------------------
# Mask
# --------------------------------------------------

target_padding_mask = create_padding_mask(
    target_token_ids,
    pad_idx=0,
)

causal_mask = create_causal_mask(
    target_len,
)

self_attention_mask = (
    target_padding_mask
    & causal_mask
)

source_padding_mask = create_padding_mask(
    source_token_ids,
    pad_idx=0,
)


# --------------------------------------------------
# Forward
# --------------------------------------------------

(
    output,
    self_attention_weights,
    cross_attention_weights,
) = decoder_layer(
    x,
    encoder_output,
    self_attention_mask,
    source_padding_mask,
)


# --------------------------------------------------
# Shape 검사
# --------------------------------------------------

assert output.shape == (
    batch_size,
    target_len,
    d_model,
)

assert self_attention_weights.shape == (
    batch_size,
    num_heads,
    target_len,
    target_len,
)

assert cross_attention_weights.shape == (
    batch_size,
    num_heads,
    target_len,
    source_len,
)


# --------------------------------------------------
# Self-Attention Mask 검사
# --------------------------------------------------

expanded_self_mask = self_attention_mask.expand(
    batch_size,
    num_heads,
    target_len,
    target_len,
)

assert torch.all(
    self_attention_weights[
        ~expanded_self_mask
    ]
    == 0
)


# --------------------------------------------------
# Source PAD Cross-Attention 검사
# --------------------------------------------------

assert torch.all(
    cross_attention_weights[
        1,
        :,
        :,
        3:
    ]
    == 0
)


# --------------------------------------------------
# dropout=0.0 반복 실행 검사
# --------------------------------------------------

(
    output2,
    self_attention_weights2,
    cross_attention_weights2,
) = decoder_layer(
    x,
    encoder_output,
    self_attention_mask,
    source_padding_mask,
)

assert torch.allclose(
    output,
    output2,
)

assert torch.allclose(
    self_attention_weights,
    self_attention_weights2,
)

assert torch.allclose(
    cross_attention_weights,
    cross_attention_weights2,
)


print(
    "DecoderLayer 통합 테스트 통과"
)

DecoderLayer 통합 테스트 통과
